In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model and tokenizer
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

# Define LoRA configuration
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # adjust if needed
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA adapters to the base model
model = get_peft_model(model, peft_config)


In [ ]:
import json
from datasets import Dataset

# Load example dataset from file
with open("concise_dataset.json", "r") as f:
    examples = json.load(f)

def format_example(example):
    return f"<|user|>\n{example['instruction']}\n<|assistant|>\n{example['response']}"

# Format and tokenize
formatted_data = [format_example(e) for e in examples]

# Tokenize dataset
tokenized = tokenizer(
    formatted_data,
    truncation=True,
    padding="max_length",
    max_length=512,
    return_tensors="pt"
)

# Wrap in Hugging Face Dataset
dataset = Dataset.from_dict({
    "input_ids": tokenized["input_ids"].tolist(),
    "attention_mask": tokenized["attention_mask"].tolist(),
    "labels": tokenized["input_ids"].tolist()
})

# Split into train and eval
split_dataset = dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Training arguments
training_args = TrainingArguments(
    output_dir="./tinyllama-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=10,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim="adamw_torch",
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# Start training
trainer.train()

In [ ]:
model.save_pretrained("./tinyllama-lora")  
tokenizer.save_pretrained("./tinyllama-lora")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
import torch

# Load tokenizer
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Inference helper
def generate_response(model, prompt, max_new_tokens=50):
    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )
    full_output = tokenizer.decode(output[0], skip_special_tokens=True)

    # Get text after <|assistant|>
    if "<|assistant|>" in full_output:
        response = full_output.split("<|assistant|>\n")[1].strip()
    else:
        response = full_output.strip()

    # Stop at first newline or period if needed
    for stop_token in ["\n", ".", "!", "?"]:
        if stop_token in response:
            response = response.split(stop_token)[0].strip()
            break

    return response

# Prompt
instruction = "Where is the capital of France?"
prompt = f"<|user|>\n{instruction}\n<|assistant|>\n"

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(model_id)

# Load LoRA-adapted model
lora_model = AutoModelForCausalLM.from_pretrained(model_id)
lora_model = PeftModel.from_pretrained(lora_model, "./tinyllama-lora")

# Generate responses
base_response = generate_response(base_model, prompt)
lora_response = generate_response(lora_model, prompt)

# Print results
print("=== Base Model Response ===")
print("Question: ",instruction)
print("Response: ",base_response)
print("\n=== LoRA-Finetuned Model Response ===")
print("Question: ",instruction)
print("Response: ",lora_response)